# Lab 8: Implementation and Performance Evaluation of Categorical Naive Bayes Classifier

## Import Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import CategoricalNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

## Load Dataset

In [2]:
# Load the dataset
dataset_path = 'lab8.csv'
df = pd.read_csv(dataset_path)

# Display required dataset properties
print("First 5 rows of the dataset:")
print(df.head())
print("-" * 50)
print(f"Dataset Shape: {df.shape}")
print("-" * 50)
print("Dataset Information:")
df.info()
print("-" * 50)
print("Missing Value Count:")
print(df.isnull().sum())

First 5 rows of the dataset:
   No   Outlook Temperature Humidity    Wind Play Tennis
0   1     Sunny         Hot     High    Weak          No
1   2     Sunny         Hot     High  Strong          No
2   3  Overcast         Hot     High    Weak         Yes
3   4      Rain        Mild     High    Weak         Yes
4   5      Rain        Cool   Normal    Weak         Yes
--------------------------------------------------
Dataset Shape: (50, 6)
--------------------------------------------------
Dataset Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   No           50 non-null     int64 
 1   Outlook      50 non-null     object
 2   Temperature  50 non-null     object
 3   Humidity     50 non-null     object
 4   Wind         50 non-null     object
 5   Play Tennis  50 non-null     object
dtypes: int64(1), object(5)
memory usage: 2.5+ KB
---

## Data Preprocessing

In [3]:
# Separate X (Input Features) and y (Target)
X = df[['Outlook', 'Temperature', 'Humidity', 'Wind']].copy()
y = df['Play Tennis'].copy()

# Initialize dictionaries to hold the LabelEncoders for feature columns
feature_encoders = {}
X_encoded = X.copy()

# Fit and transform each feature column with its own LabelEncoder
for col in X.columns:
    le = LabelEncoder()
    X_encoded[col] = le.fit_transform(X[col])
    feature_encoders[col] = le
    print(f"Mapping for '{col}':")
    for class_index, class_label in enumerate(le.classes_):
        print(f"  {class_label} -> {class_index}")

# Initialize and fit a LabelEncoder for the target column
target_encoder = LabelEncoder()
y_encoded = target_encoder.fit_transform(y)
print("\nMapping for Target 'Play':")
for class_index, class_label in enumerate(target_encoder.classes_):
        print(f"  {class_label} -> {class_index}")

# Print the encoded dataset
encoded_df = X_encoded.copy()
encoded_df['Play'] = y_encoded
print("-" * 50)
print("Encoded Dataset (First 10 rows):")
print(encoded_df.head(10))
print("-" * 50)
print("Pre-processing and encoding completed successfully.")

Mapping for 'Outlook':
  Overcast -> 0
  Rain -> 1
  Sunny -> 2
Mapping for 'Temperature':
  Cool -> 0
  Hot -> 1
  Mild -> 2
Mapping for 'Humidity':
  High -> 0
  Normal -> 1
Mapping for 'Wind':
  Strong -> 0
  Weak -> 1

Mapping for Target 'Play':
  No -> 0
  Yes -> 1
--------------------------------------------------
Encoded Dataset (First 10 rows):
   Outlook  Temperature  Humidity  Wind  Play
0        2            1         0     1     0
1        2            1         0     0     0
2        0            1         0     1     1
3        1            2         0     1     1
4        1            0         1     1     1
5        1            0         1     0     0
6        0            0         1     0     1
7        2            2         0     1     0
8        2            0         1     1     1
9        1            2         1     1     1
--------------------------------------------------
Pre-processing and encoding completed successfully.


## Dataset Partitioning

In [4]:
# Split the dataset: 80% Training, 20% Testing
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, 
    y_encoded, 
    test_size=0.20, 
    random_state=42, 
    stratify=y_encoded
)

print(f"Training Features Shape: {X_train.shape}")
print(f"Training Target Shape:   {y_train.shape}")
print(f"Testing Features Shape:  {X_test.shape}")
print(f"Testing Target Shape:    {y_test.shape}")

Training Features Shape: (40, 4)
Training Target Shape:   (40,)
Testing Features Shape:  (10, 4)
Testing Target Shape:    (10,)


## Categorical Naive Bayes Model Training

In [5]:
# Initialize and train CategoricalNB
nb_model = CategoricalNB()
nb_model.fit(X_train, y_train)

,alpha,1.0
,force_alpha,True
,fit_prior,True
,class_prior,None
,min_categories,None


## Model Evaluation

In [6]:
# Predict labels for testing dataset
y_pred_nb = nb_model.predict(X_test)

# Calculate evaluation metrics
accuracy_nb = accuracy_score(y_test, y_pred_nb)
conf_matrix_nb = confusion_matrix(y_test, y_pred_nb)
class_report_nb = classification_report(
    y_test, 
    y_pred_nb, 
    target_names=target_encoder.classes_
)

print(f"Categorical Naive Bayes Test Accuracy: {accuracy_nb:.4f}")
print("\nConfusion Matrix:")
print(conf_matrix_nb)
print("\nClassification Report:")
print(class_report_nb)

Categorical Naive Bayes Test Accuracy: 0.9000

Confusion Matrix:
[[2 1]
 [0 7]]

Classification Report:
              precision    recall  f1-score   support

          No       1.00      0.67      0.80         3
         Yes       0.88      1.00      0.93         7

    accuracy                           0.90        10
   macro avg       0.94      0.83      0.87        10
weighted avg       0.91      0.90      0.89        10



## Single-Sample Inference

In [7]:
# Weather conditions: Sunny, Cool, High, Strong
sample_input = {
    'Outlook': 'Sunny',
    'Temperature': 'Cool',
    'Humidity': 'High',
    'Wind': 'Strong'
}

print(f"Input Weather Conditions: {sample_input}")

# Encode the sample using the stored LabelEncoders
encoded_sample = []
for col in ['Outlook', 'Temperature', 'Humidity', 'Wind']:
    le = feature_encoders[col]
    val_encoded = le.transform([sample_input[col]])[0]
    encoded_sample.append(val_encoded)

# Convert to a DataFrame with feature names to avoid warnings during prediction
encoded_sample_df = pd.DataFrame([encoded_sample], columns=X_encoded.columns)
print(f"Encoded Sample Vector:    {encoded_sample}")

# Predict class label and class probabilities
pred_class_encoded = nb_model.predict(encoded_sample_df)[0]
pred_proba = nb_model.predict_proba(encoded_sample_df)[0]

# Map probabilities to classes
classes = target_encoder.classes_
no_idx = list(classes).index('No')
yes_idx = list(classes).index('Yes')

prob_no = pred_proba[no_idx]
prob_yes = pred_proba[yes_idx]

# Inverse transform to get original text label
pred_class_text = target_encoder.inverse_transform([pred_class_encoded])[0]

print("-" * 50)
print(f"Predicted Class Label:        {pred_class_text}")
print(f"Probability of Play = Yes:     {prob_yes:.4f}")
print(f"Probability of Play = No:      {prob_no:.4f}")
print("-" * 50)

Input Weather Conditions: {'Outlook': 'Sunny', 'Temperature': 'Cool', 'Humidity': 'High', 'Wind': 'Strong'}
Encoded Sample Vector:    [np.int64(2), np.int64(0), np.int64(0), np.int64(0)]
--------------------------------------------------
Predicted Class Label:        No
Probability of Play = Yes:     0.2106
Probability of Play = No:      0.7894
--------------------------------------------------


## Model Comparison (Decision Tree, Logistic Regression, SVM)

In [8]:
# 1. Decision Tree Classifier
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)
print("1. Decision Tree Classifier trained.")

# 2. Logistic Regression (max_iter=1000)
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)
print("2. Logistic Regression trained.")

# 3. Support Vector Machine (SVC with probability=True)
svm_model = SVC(probability=True, random_state=42)
svm_model.fit(X_train, y_train)
print("3. Support Vector Machine (SVM) trained.")

1. Decision Tree Classifier trained.
2. Logistic Regression trained.
3. Support Vector Machine (SVM) trained.


## Comparison Table

In [9]:
models = {
    'Categorical Naive Bayes': nb_model,
    'Decision Tree': dt_model,
    'Logistic Regression': lr_model,
    'SVM': svm_model
}

comparison_data = []

for name, model in models.items():
    # 1. Test Accuracy
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    
    # 2. Prediction for the sample
    sample_pred_encoded = model.predict(encoded_sample_df)[0]
    sample_pred_text = target_encoder.inverse_transform([sample_pred_encoded])[0]
    
    # 3. Prediction Probabilities
    try:
        sample_proba = model.predict_proba(encoded_sample_df)[0]
        p_no = sample_proba[no_idx]
        p_yes = sample_proba[yes_idx]
        proba_str = f"Yes: {p_yes:.4f}, No: {p_no:.4f}"
    except AttributeError:
        p_no = np.nan
        p_yes = np.nan
        proba_str = "N/A - Model does not support probability estimates directly."
        print(f"Note: {name} does not support probability estimation by default.")
    
    # Print individual model evaluation details
    print(f"--- Model: {name} ---")
    print(f"  Test Accuracy:               {accuracy:.4f}")
    print(f"  Sample Prediction:           {sample_pred_text}")
    print(f"  Sample Probabilities:        {proba_str}")
    print("-" * 50)
    
    # Save to comparison list
    comparison_data.append({
        'Model Name': name,
        'Test Accuracy': round(accuracy, 4),
        'Predicted Class': sample_pred_text,
        'Probability(No)': round(p_no, 4) if not np.isnan(p_no) else "N/A",
        'Probability(Yes)': round(p_yes, 4) if not np.isnan(p_yes) else "N/A"
    })

# Create the comparison table DataFrame
comparison_df = pd.DataFrame(comparison_data)

print("\n=========================")
print("COMPARISON TABLE")
print("=========================")
print(comparison_df.to_string(index=False))

--- Model: Categorical Naive Bayes ---
  Test Accuracy:               0.9000
  Sample Prediction:           No
  Sample Probabilities:        Yes: 0.2106, No: 0.7894
--------------------------------------------------
--- Model: Decision Tree ---
  Test Accuracy:               1.0000
  Sample Prediction:           No
  Sample Probabilities:        Yes: 0.0000, No: 1.0000
--------------------------------------------------
--- Model: Logistic Regression ---
  Test Accuracy:               0.9000
  Sample Prediction:           No
  Sample Probabilities:        Yes: 0.1387, No: 0.8613
--------------------------------------------------
--- Model: SVM ---
  Test Accuracy:               1.0000
  Sample Prediction:           No
  Sample Probabilities:        Yes: 0.0431, No: 0.9569
--------------------------------------------------

COMPARISON TABLE
             Model Name  Test Accuracy Predicted Class  Probability(No)  Probability(Yes)
Categorical Naive Bayes            0.9              No    

## Analysis Report

In [10]:
analysis_report = (
    "Naive Bayes, Decision Tree, Logistic Regression, and SVM may produce different predictions and probability estimates "
    "due to their distinct underlying algorithmic architectures and mathematical formulations. "
    "Naive Bayes assumes conditional independence among input features given the class label, multiplying individual probabilities. "
    "In contrast, Decision Trees partition the feature space into hierarchical, orthogonal decision boundaries based on information gain. "
    "Logistic Regression models the probability using a logistic sigmoid function over a linear decision boundary, whereas "
    "SVM focuses on maximizing the geometric margin between classes and relies on scaling techniques (Platt calibration) "
    "to estimate posterior class probabilities."
)

print(analysis_report)

Naive Bayes, Decision Tree, Logistic Regression, and SVM may produce different predictions and probability estimates due to their distinct underlying algorithmic architectures and mathematical formulations. Naive Bayes assumes conditional independence among input features given the class label, multiplying individual probabilities. In contrast, Decision Trees partition the feature space into hierarchical, orthogonal decision boundaries based on information gain. Logistic Regression models the probability using a logistic sigmoid function over a linear decision boundary, whereas SVM focuses on maximizing the geometric margin between classes and relies on scaling techniques (Platt calibration) to estimate posterior class probabilities.
